# 🔋 Battery State-of-Health (SoH) Estimation — Week 3

**WIDSS — Windowed Intelligent Drive-cycle State eStimation**

This notebook covers:
1. Synthetic EV drive-cycle simulation using the `widss` package
2. Exploratory Data Analysis (EDA)
3. Feature engineering for SoC estimation
4. Baseline ML models (Linear Regression, Random Forest)
5. LSTM model training and evaluation

> **Context**: Battery SoH describes long-term degradation (capacity fade). SoC is the instantaneous charge level.  
> In this week we focus on SoC estimation as the foundation before extending to SoH.


## 0. Environment Setup

In [ ]:
# Install WIDSS (run once)
# !pip install -e ".[tensorflow]"  # from repo root

# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11
})
print('✅ Core imports done')

In [ ]:
# WIDSS imports
import sys, os
sys.path.insert(0, os.path.abspath('../src'))  # adjust if running from repo root

from widss.simulation import BatterySimulationConfig, build_dataset
from widss.dataset import build_sequences
from widss.evaluation import rmse, mae, mape

print('✅ WIDSS package loaded')

---
## 1. Simulate EV Drive Cycle Data

In [ ]:
# Simulate 2 hours of EV driving
SEED = 42
DURATION_S = 7200  # 2 hours

cfg = BatterySimulationConfig(
    capacity_ah=60.0,
    soc_init=0.95,
    dt_s=1.0,
    internal_resistance_ohm=0.02,
    ocv_min_v=3.0,
    ocv_max_v=4.2
)

df = build_dataset(duration_s=DURATION_S, config=cfg, seed=SEED)
print(f'Dataset shape : {df.shape}')
print(f'Columns       : {list(df.columns)}')
print(f'SoC range     : [{df.soc.min():.3f}, {df.soc.max():.3f}]')
df.head(10)

---
## 2. Exploratory Data Analysis (EDA)

In [ ]:
# ── Statistical summary ──────────────────────────────────────────────────────
df.describe().round(4)

In [ ]:
# ── Time-series overview ─────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

t = df['time_s'] / 3600  # hours

axes[0].plot(t, df['current_a'], color='steelblue', lw=0.8, label='Current (A)')
axes[0].axhline(0, color='k', lw=0.5, ls='--')
axes[0].set_ylabel('Current (A)')
axes[0].set_title('EV Drive Cycle — Simulated Battery Data')
axes[0].legend()

axes[1].plot(t, df['voltage_v'], color='darkorange', lw=0.8, label='Terminal Voltage (V)')
axes[1].set_ylabel('Voltage (V)')
axes[1].legend()

axes[2].plot(t, df['soc'], color='green', lw=1.2, label='SoC')
axes[2].set_xlabel('Time (hours)')
axes[2].set_ylabel('SoC')
axes[2].set_ylim(0, 1.05)
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Distribution plots ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, color, title in zip(
    axes,
    ['current_a', 'voltage_v', 'soc'],
    ['steelblue', 'darkorange', 'green'],
    ['Current Distribution', 'Voltage Distribution', 'SoC Distribution']
):
    ax.hist(df[col], bins=60, color=color, alpha=0.75, edgecolor='white')
    ax.set_title(title)
    ax.set_xlabel(col)
    ax.set_ylabel('Count')

plt.suptitle('Feature Distributions', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation matrix ───────────────────────────────────────────────────────
import seaborn as sns

corr = df[['current_a', 'voltage_v', 'soc']].corr()
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# ── OCV–SoC relationship (key physics relationship) ──────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
sc = ax.scatter(df['soc'], df['voltage_v'], c=df['current_a'],
                cmap='RdBu_r', alpha=0.3, s=2)
plt.colorbar(sc, label='Current (A) — blue=charging, red=discharging')
ax.set_xlabel('State of Charge')
ax.set_ylabel('Terminal Voltage (V)')
ax.set_title('OCV–SoC Relationship (colored by current)')
plt.tight_layout()
plt.show()

---
## 3. Feature Engineering

In [ ]:
# Add derived features useful for SoC estimation
df_feat = df.copy()

# Rolling statistics (1-minute window = 60 s at 1 Hz)
WIN = 60
df_feat['current_roll_mean'] = df_feat['current_a'].rolling(WIN, min_periods=1).mean()
df_feat['current_roll_std']  = df_feat['current_a'].rolling(WIN, min_periods=1).std().fillna(0)
df_feat['voltage_roll_mean'] = df_feat['voltage_v'].rolling(WIN, min_periods=1).mean()

# Cumulative charge (Coulomb counting) — key physics feature
df_feat['charge_ah'] = (df_feat['current_a'] * cfg.dt_s / 3600).cumsum()

# Power
df_feat['power_w'] = df_feat['current_a'] * df_feat['voltage_v']

# Lag features
for lag in [1, 5, 30]:
    df_feat[f'current_lag{lag}'] = df_feat['current_a'].shift(lag).fillna(0)
    df_feat[f'voltage_lag{lag}'] = df_feat['voltage_v'].shift(lag).fillna(method='bfill')

print(f'Features after engineering: {list(df_feat.columns)}')
print(f'Shape: {df_feat.shape}')
df_feat.head()

---
## 4. Baseline Models — Train/Test Split

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

FEATURE_COLS = [
    'current_a', 'voltage_v',
    'current_roll_mean', 'current_roll_std', 'voltage_roll_mean',
    'charge_ah', 'power_w',
    'current_lag1', 'voltage_lag1',
    'current_lag5', 'voltage_lag5',
    'current_lag30', 'voltage_lag30'
]
TARGET = 'soc'

# 80/20 train-test split (time-ordered — no shuffle!)
split_idx = int(0.8 * len(df_feat))
train = df_feat.iloc[:split_idx]
test  = df_feat.iloc[split_idx:]

X_train = train[FEATURE_COLS].values
y_train = train[TARGET].values
X_test  = test[FEATURE_COLS].values
y_test  = test[TARGET].values

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

In [ ]:
# ── Train & evaluate baseline models ─────────────────────────────────────────
results = {}

models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression' : Ridge(alpha=1.0),
    'Random Forest'    : RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, learning_rate=0.05,
                                                    max_depth=4, random_state=42)
}

for name, model in models.items():
    model.fit(X_train_sc, y_train)
    y_pred = model.predict(X_test_sc)
    y_pred = np.clip(y_pred, 0, 1)  # SoC is bounded [0, 1]
    results[name] = {
        'RMSE' : rmse(y_test, y_pred),
        'MAE'  : mae(y_test, y_pred),
        'MAPE' : mape(y_test, y_pred),
        'pred' : y_pred
    }
    print(f'{name:22s}  RMSE={results[name]["RMSE"]:.4f}  MAE={results[name]["MAE"]:.4f}  MAPE={results[name]["MAPE"]:.2f}%')

In [ ]:
# ── Results comparison table ─────────────────────────────────────────────────
metrics_df = pd.DataFrame(
    {name: {k: v for k, v in vals.items() if k != 'pred'}
     for name, vals in results.items()}
).T.round(4)
metrics_df.sort_values('RMSE')

In [ ]:
# ── Prediction plots ─────────────────────────────────────────────────────────
t_test = test['time_s'].values / 3600

fig, axes = plt.subplots(2, 2, figsize=(15, 8), sharex=True, sharey=True)
axes = axes.flatten()

for ax, (name, vals) in zip(axes, results.items()):
    ax.plot(t_test, y_test, 'k-', lw=1.5, label='True SoC', alpha=0.7)
    ax.plot(t_test, vals['pred'], '--', lw=1.2, label=f'Predicted (RMSE={vals["RMSE"]:.4f})')
    ax.set_title(name)
    ax.set_xlabel('Time (hours)')
    ax.set_ylabel('SoC')
    ax.legend(fontsize=9)
    ax.set_ylim(0, 1.05)

plt.suptitle('Baseline Model Predictions vs True SoC', fontsize=13)
plt.tight_layout()
plt.show()

---
## 5. Feature Importance (Random Forest)

In [ ]:
rf_model = models['Random Forest']
importances = rf_model.feature_importances_
feat_imp = pd.Series(importances, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
feat_imp.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Random Forest — Feature Importances')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

---
## 6. LSTM Sequence Model

We now use the `widss` LSTM pipeline which operates on raw sliding windows of current/voltage.

In [ ]:
from widss.model import build_lstm_soc_model, tensorflow_available

WINDOW_SIZE = 30

if not tensorflow_available():
    print('⚠️  TensorFlow not installed — skipping LSTM training.')
    print('    Run: pip install tensorflow>=2.13')
else:
    import tensorflow as tf
    print(f'TensorFlow version: {tf.__version__}')

    # Build sequences
    X_seq, y_seq = build_sequences(df, window_size=WINDOW_SIZE)
    split = int(0.8 * len(X_seq))
    X_tr, X_te = X_seq[:split], X_seq[split:]
    y_tr, y_te = y_seq[:split], y_seq[split:]
    print(f'Train sequences: {X_tr.shape}  |  Test sequences: {X_te.shape}')

In [ ]:
if tensorflow_available():
    model_lstm = build_lstm_soc_model(
        window_size=WINDOW_SIZE,
        feature_count=2,
        units=64,
        learning_rate=1e-3
    )
    model_lstm.summary()
    
    history = model_lstm.fit(
        X_tr, y_tr,
        validation_split=0.15,
        epochs=10,
        batch_size=64,
        verbose=1
    )

In [ ]:
if tensorflow_available():
    # Training curves
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history['loss'], label='Train Loss')
    axes[0].plot(history.history['val_loss'], label='Val Loss')
    axes[0].set_title('Loss (MSE)')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()

    axes[1].plot(history.history.get('mae', []), label='Train MAE')
    axes[1].plot(history.history.get('val_mae', []), label='Val MAE')
    axes[1].set_title('MAE')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()

    plt.suptitle('LSTM Training Curves', fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
if tensorflow_available():
    # Evaluate LSTM
    y_lstm_pred = model_lstm.predict(X_te, verbose=0).flatten()
    y_lstm_pred = np.clip(y_lstm_pred, 0, 1)

    print('── LSTM Results ─────────────────────────────────────────')
    print(f'RMSE : {rmse(y_te, y_lstm_pred):.4f}')
    print(f'MAE  : {mae(y_te, y_lstm_pred):.4f}')
    print(f'MAPE : {mape(y_te, y_lstm_pred):.2f} %')

    # Prediction plot
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(y_te[:500], 'k-', lw=1.5, label='True SoC')
    ax.plot(y_lstm_pred[:500], 'r--', lw=1.2, label='LSTM Predicted')
    ax.set_xlabel('Timestep')
    ax.set_ylabel('SoC')
    ax.set_title('LSTM — SoC Prediction (first 500 test steps)')
    ax.legend()
    plt.tight_layout()
    plt.show()

---
## 7. Summary

In [ ]:
print('Week 3 Summary')
print('='*55)
print(f'Dataset         : {DURATION_S/3600:.0f}-hour synthetic EV drive cycle')
print(f'Battery config  : {cfg.capacity_ah} Ah Li-ion')
print(f'Features used   : {len(FEATURE_COLS)} engineered features')
print()
print('Baseline Results (Test Set):')
for name, vals in results.items():
    print(f'  {name:22s} → RMSE {vals["RMSE"]:.4f}, MAE {vals["MAE"]:.4f}')

print()
print('Next Steps (Week 3 Updated):')
print('  • Hyperparameter tuning for Random Forest / GBM')
print('  • Deeper LSTM (stacked layers, dropout regularisation)')
print('  • Multiple battery configurations / seed robustness')